# PCA of prompt activations by subset and layer

Select a saved dataset subset in the **selection cell**, then run the remaining cells.
This notebook computes the top three principal components independently for **every
extracted layer**, using every prompt in that subset. The interactive 3D and 2D plots each have a
layer dropdown; hover over a point for its original prompt, labels, and selected token.

The completed Llama-3.1-8B extraction is loaded from
`dalg-cache/sae_manifold_llama31_8b_last`. Each point represents one
prompt using its **last token**, at the output of a zero-based decoder
block before final normalization. The language model does not need to
be loaded again; the extraction must already be complete.

PCA centers the activations but does not standardize features or whiten scores.
Each layer has its own fitted basis: axis signs/orientations are not aligned between
layers. See [the extraction contract](../docs/reference/prompt-activations.md).

In [ ]:
from pathlib import Path
from collections import defaultdict
from html import escape
import json
import os
import sys
from time import perf_counter

import numpy as np
import pandas as pd
import torch
from sklearn.decomposition import PCA
from joblib import Parallel, delayed, parallel_config
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
from IPython.display import display

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))
from dalg.data.prompt_activations import load_prompt_activation_index

pio.renderers.default = "plotly_mimetype"
ACTIVATION_DIR = REPO_ROOT / "dalg-cache/sae_manifold_llama31_8b_last"
config = json.loads((ACTIVATION_DIR / "config.json").read_text())
LAYERS = sorted(config["layers"])
# This reader checks completion and joins source records to activation coordinates.
index = load_prompt_activation_index(ACTIVATION_DIR, layer=LAYERS[0])
metadata = pd.DataFrame(index)
SUBSETS = sorted(metadata["subset"].unique())
summary = metadata.groupby("subset").size().rename("prompts").to_frame()
summary["label_fields"] = [
    sorted({key for labels in metadata.loc[metadata.subset.eq(name), "labels"]
            for key in labels})
    for name in summary.index
]
print(f"{config['model']}: {len(metadata):,} prompts, {len(LAYERS)} layers, "
      f"{config['d_model']} activation features; last token")
display(summary)

## Select a subset

Edit `SUBSET` to any name in the table above and rerun this cell and everything below.
`COLOR_LABEL = None` chooses a useful ground-truth label automatically; set it to
another field from the table to change the coloring. For `colors`, `"hex"` uses the
actual saved colors. Changing only the coloring requires rerunning this cell and
the plotting cells, without recomputing PCA.

In [ ]:
SUBSET = "years"  # colors, age, temperature, geography, days, years, formality, sent_length
COLOR_LABEL = None
SEED = 0
N_WORKERS = min(8, len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity")
                else (os.cpu_count() or 1))  # Increase to run more layers concurrently.

if SUBSET not in SUBSETS:
    raise ValueError(f"Choose SUBSET from {SUBSETS}")
selected = metadata.loc[metadata.subset.eq(SUBSET)].reset_index(drop=True)
labels = pd.json_normalize(selected["labels"].tolist())
DEFAULT_COLOR = {
    "colors": "hue", "age": "age", "temperature": "fahrenheit",
    "geography": "continent", "days": "day_idx", "years": "year",
    "formality": "formality", "sent_length": "n_tokens",
}
color_label = COLOR_LABEL or DEFAULT_COLOR.get(SUBSET, labels.columns[0])
if color_label not in labels:
    raise ValueError(f"Choose COLOR_LABEL from {labels.columns.tolist()}")
print(f"Selected {SUBSET}: {len(selected):,} prompts; color = {color_label}")
display(pd.concat([selected[["source_row_id", "prompt", "decoded_token"]], labels], axis=1).head())

## Fit three PCs for every layer in parallel

`N_WORKERS` controls how many layers run concurrently (up to eight by default).
Each worker loads only the selected subset, uses one CPU thread, and fits all its
rows, including distinct records with duplicate prompt text. A seeded randomized
SVD estimates the top three PCs with 20 oversamples and 7 power iterations.

Completed layers report loading and PCA times immediately, in completion order.
The final tables and layer selector remain ordered by layer number. Each worker
holds one activation matrix, so memory use increases with `N_WORKERS`.
`results` retains scores, PCA models, and matching source row IDs. No extraction
artifacts are modified.

In [10]:
def load_subset_layer(rows, layer):
    """Read selected rows in their metadata order, grouping reads by saved shard."""
    groups = defaultdict(list)
    for output_row, row in enumerate(rows.itertuples(index=False)):
        groups[int(row.shard)].append((output_row, int(row.row_in_shard), int(row.token_slot)))
    values = np.empty((len(rows), config["d_model"]), dtype=np.float32)
    for shard_id, coordinates in groups.items():
        path = ACTIVATION_DIR / f"layer{layer:02d}" / f"shard_{shard_id:05d}.pt"
        shard = torch.load(path, map_location="cpu", weights_only=True, mmap=True)
        if shard.ndim != 3 or tuple(shard.shape[1:]) != (1, config["d_model"]):
            raise ValueError(f"Unexpected activation shape in {path}: {tuple(shard.shape)}")
        output_rows, shard_rows, token_slots = map(list, zip(*coordinates))
        values[output_rows] = shard[shard_rows, token_slots].float().numpy()
    if not np.isfinite(values).all():
        raise ValueError(f"Nonfinite activations in layer {layer}")
    return values



def fit_layer(layer, rows, subset, seed):
    """Load and project one layer; joblib limits each worker to one CPU thread."""
    started = perf_counter()
    x = load_subset_layer(rows, layer)
    loaded = perf_counter()
    if not np.any(x != x[0]):
        raise ValueError(f"Layer {layer} has zero variance across the selected prompts")
    pca = PCA(n_components=3, svd_solver="randomized", random_state=seed,
              n_oversamples=20, iterated_power=7)
    scores = pca.fit_transform(x)
    fitted = perf_counter()
    result = {
        "scores": scores, "pca": pca, "subset": subset, "activation_dir": str(ACTIVATION_DIR),
        "source_row_ids": rows["source_row_id"].to_numpy(copy=True),
        "load_seconds": loaded - started, "pca_seconds": fitted - loaded,
    }
    return layer, result


if min(len(selected) - 1, config["d_model"]) < 3:
    raise ValueError("Three centered PCs require at least four prompts and three features")
if not isinstance(N_WORKERS, int) or N_WORKERS < 1:
    raise ValueError("N_WORKERS must be a positive integer")
results = {}
workers = min(N_WORKERS, len(LAYERS))
started = perf_counter()
print(f"Fitting {len(LAYERS)} layers with {workers} parallel workers...", flush=True)
# Separate processes avoid competing BLAS/OpenMP thread pools in the kernel.
with parallel_config(backend="loky", inner_max_num_threads=1):
    completed = Parallel(n_jobs=workers, return_as="generator_unordered")(
        delayed(fit_layer)(layer, selected, SUBSET, SEED) for layer in LAYERS
    )
    for layer, result in completed:
        results[layer] = result
        ratios = result["pca"].explained_variance_ratio_
        print(f"[{len(results):02d}/{len(LAYERS)}] Layer {layer:02d}: "
              f"top 3 explain {ratios.sum():.2%}; "
              f"load {result['load_seconds']:.2f}s, PCA {result['pca_seconds']:.2f}s",
              flush=True)
results = dict(sorted(results.items()))
variance = pd.DataFrame([
    {"layer": layer, "PC1": result["pca"].explained_variance_ratio_[0],
     "PC2": result["pca"].explained_variance_ratio_[1],
     "PC3": result["pca"].explained_variance_ratio_[2],
     "top3": result["pca"].explained_variance_ratio_.sum()}
    for layer, result in results.items()
]).set_index("layer")
print(f"Finished in {perf_counter() - started:.1f}s", flush=True)

Fitting 32 layers with 8 parallel workers...
[01/32] Layer 00: top 3 explain 18.81%; load 0.10s, PCA 0.02s
[02/32] Layer 02: top 3 explain 21.51%; load 0.13s, PCA 0.02s
[03/32] Layer 05: top 3 explain 25.57%; load 0.09s, PCA 0.05s
[04/32] Layer 07: top 3 explain 32.83%; load 0.10s, PCA 0.05s
[05/32] Layer 06: top 3 explain 29.71%; load 0.15s, PCA 0.02s
[06/32] Layer 01: top 3 explain 21.10%; load 0.17s, PCA 0.03s
[07/32] Layer 08: top 3 explain 28.41%; load 0.06s, PCA 0.03s
[08/32] Layer 04: top 3 explain 25.44%; load 0.16s, PCA 0.04s
[09/32] Layer 03: top 3 explain 24.58%; load 0.18s, PCA 0.02s
[10/32] Layer 09: top 3 explain 28.88%; load 0.10s, PCA 0.02s
[11/32] Layer 10: top 3 explain 28.85%; load 0.12s, PCA 0.04s
[12/32] Layer 13: top 3 explain 28.66%; load 0.09s, PCA 0.04s
[13/32] Layer 11: top 3 explain 28.49%; load 0.15s, PCA 0.04s
[14/32] Layer 12: top 3 explain 27.01%; load 0.16s, PCA 0.04s
[15/32] Layer 14: top 3 explain 26.12%; load 0.16s, PCA 0.04s
[16/32] Layer 16: top 3 e

## Interactive 3D projection

Rotate, zoom, and select any layer from the dropdown. Axis labels report the
variance explained by each PC. Every selected prompt appears in the plot; hover
text preserves whitespace with `repr`. Color is shared across layers. A changed
subset requires a fresh PCA fit before plotting.

In [ ]:
for result in results.values():
    if result["activation_dir"] != str(ACTIVATION_DIR) or result["subset"] != SUBSET or not np.array_equal(
        result["source_row_ids"], selected["source_row_id"].to_numpy()
    ):
        raise ValueError("The selection changed: rerun the PCA fitting cell")

def token_description(row):
    return f"token={escape(repr(row.decoded_token))} (index {row.token_index})"


hover_text = [
    f"row_id={row.source_row_id}<br>prompt={escape(repr(row.prompt))}"
    f"<br>{token_description(row)}"
    f"<br>labels={escape(json.dumps(row.labels, ensure_ascii=False))}"
    for row in selected.itertuples(index=False)
]
marker = {"size": 3, "opacity": 0.8}
color_values = labels[color_label]
numeric = pd.to_numeric(color_values, errors="coerce")
if color_label == "hex":
    marker["color"] = color_values.tolist()
elif numeric.notna().all():
    marker.update(color=numeric.to_numpy(), colorscale="Viridis", showscale=True,
                  colorbar={"title": color_label})
else:
    categories = sorted(color_values.fillna("(missing)").astype(str).unique())
    codes = pd.Categorical(color_values.fillna("(missing)").astype(str), categories).codes
    palette = px.colors.qualitative.Alphabet
    scale = []
    for i in range(len(categories)):
        color = palette[i % len(palette)]
        scale.extend([(i / len(categories), color), ((i + 1) / len(categories), color)])
    marker.update(color=codes, colorscale=scale, cmin=-0.5, cmax=len(categories) - 0.5,
                  showscale=True, colorbar={"title": color_label,
                  "tickvals": list(range(len(categories))), "ticktext": categories})


def layer_layout(layer):
    ratios = results[layer]["pca"].explained_variance_ratio_
    return {
        "title": {"text": f"{SUBSET} · last token · layer {layer:02d} · top 3 explain {ratios.sum():.1%}"},
        "scene": {"xaxis_title": f"PC1 ({ratios[0]:.1%})",
                  "yaxis_title": f"PC2 ({ratios[1]:.1%})",
                  "zaxis_title": f"PC3 ({ratios[2]:.1%})", "aspectmode": "data"},
    }

first = results[LAYERS[0]]["scores"]
fig = go.Figure(go.Scatter3d(
    x=first[:, 0], y=first[:, 1], z=first[:, 2], mode="markers", marker=marker,
    text=hover_text,
    hovertemplate="%{text}<br>PC1=%{x:.3f}<br>PC2=%{y:.3f}<br>PC3=%{z:.3f}<extra></extra>",
))
buttons = []
for layer in LAYERS:
    scores = results[layer]["scores"]
    buttons.append({"label": f"Layer {layer:02d}", "method": "update", "args": [
        {"x": [scores[:, 0]], "y": [scores[:, 1]], "z": [scores[:, 2]]},
        layer_layout(layer),
    ]})
fig.update_layout(**layer_layout(LAYERS[0]), height=720,
                  margin={"l": 0, "r": 0, "b": 0, "t": 100},
                  updatemenus=[{"buttons": buttons, "x": 0, "y": 1.10,
                                "xanchor": "left", "yanchor": "top"}])
fig.show()

## Interactive 2D projection

View PC1 against PC2 with the same layer selector, colors, and prompt hover text.
Run the 3D plotting cell above first to prepare the shared colors and hover text.
This view reuses the fitted PCA scores without recomputing PCA. The title reports
the variance retained by these two PCs; equal axis scaling preserves distances
in the projected plane.


In [ ]:
for result in results.values():
    if result["activation_dir"] != str(ACTIVATION_DIR) or result["subset"] != SUBSET or not np.array_equal(
        result["source_row_ids"], selected["source_row_id"].to_numpy()
    ):
        raise ValueError("The selection changed: rerun the PCA fitting and 3D plotting cells")


def layer_layout_2d(layer):
    ratios = results[layer]["pca"].explained_variance_ratio_
    return {
        "title": {"text": f"{SUBSET} · last token · layer {layer:02d} · top 2 explain {ratios[:2].sum():.1%}"},
        "xaxis": {"title": f"PC1 ({ratios[0]:.1%})", "autorange": True},
        "yaxis": {"title": f"PC2 ({ratios[1]:.1%})", "autorange": True,
                  "scaleanchor": "x", "scaleratio": 1},
    }


first_2d = results[LAYERS[0]]["scores"]
fig_2d = go.Figure(go.Scattergl(
    x=first_2d[:, 0], y=first_2d[:, 1], mode="markers",
    marker={**marker, "size": 6}, text=hover_text,
    hovertemplate="%{text}<br>PC1=%{x:.3f}<br>PC2=%{y:.3f}<extra></extra>",
))
buttons_2d = []
for layer in LAYERS:
    scores = results[layer]["scores"]
    buttons_2d.append({"label": f"Layer {layer:02d}", "method": "update", "args": [
        {"x": [scores[:, 0]], "y": [scores[:, 1]]},
        layer_layout_2d(layer),
    ]})
fig_2d.update_layout(**layer_layout_2d(LAYERS[0]), height=720,
                     margin={"l": 60, "r": 30, "b": 60, "t": 100},
                     updatemenus=[{"buttons": buttons_2d, "x": 0, "y": 1.10,
                                   "xanchor": "left", "yanchor": "top"}])
fig_2d.show()


## Explained variance across layers

These fractions measure how much of the selected subset's centered activation
variance the projection retains. A low top-three total means substantial geometry
is omitted from the 3D view.

In [ ]:
variance_fig = go.Figure()
for component in ["PC1", "PC2", "PC3", "top3"]:
    variance_fig.add_scatter(x=variance.index, y=variance[component], mode="lines+markers",
                             name="Top 3 total" if component == "top3" else component)
variance_fig.update_layout(title=f"{SUBSET} · last token: variance explained by layer", height=380,
                           xaxis_title="Decoder block (zero-based)",
                           yaxis_title="Explained variance fraction", yaxis_tickformat=".0%",
                           yaxis_range=[0, 1])
variance_fig.show()
display(variance.style.format("{:.2%}"))

## UMAP — layer 25 only

This section fits a **3D UMAP** to every selected prompt's original 4,096-dimensional
activation at **zero-based layer 25**. It uses Euclidean distances, without feature
standardization or projecting onto the three PCA axes first. Labels are used only
for coloring, never for fitting.

Run the preceding cells once, then run these cells to add UMAP without rerunning
the layer PCA fits. `UMAP_N_NEIGHBORS` and `UMAP_MIN_DIST` control the embedding.
The fixed `SEED` makes the fit reproducible; UMAP uses one worker for this fit.
The first call may take longer while Numba compiles its numerical routines.
UMAP axes have no explained-variance interpretation.

In [14]:
print("Importing UMAP...", flush=True)
from umap import UMAP

UMAP_LAYER = 19
UMAP_N_NEIGHBORS = 15
UMAP_MIN_DIST = 0.1

if UMAP_LAYER not in LAYERS:
    raise ValueError(f"Layer {UMAP_LAYER} was not extracted")
if len(selected) < 5:
    raise ValueError("3D UMAP with spectral initialization requires at least five prompts")

print(f"Loading {SUBSET}, layer {UMAP_LAYER} for UMAP...", flush=True)
umap_started = perf_counter()
umap_x = load_subset_layer(selected, UMAP_LAYER)
umap_loaded = perf_counter()
umap_model = UMAP(
    n_components=3,
    n_neighbors=min(UMAP_N_NEIGHBORS, len(selected) - 1),
    min_dist=UMAP_MIN_DIST,
    metric="euclidean",
    random_state=SEED,
    n_jobs=1,
)
print(f"Loaded in {umap_loaded - umap_started:.2f}s; fitting 3D UMAP...", flush=True)
umap_scores = umap_model.fit_transform(umap_x)
umap_result = {
    "scores": umap_scores,
    "model": umap_model,
    "subset": SUBSET,
    "activation_dir": str(ACTIVATION_DIR),
    "layer": UMAP_LAYER,
    "source_row_ids": selected["source_row_id"].to_numpy(copy=True),
}
del umap_x
print(f"UMAP fit finished in {perf_counter() - umap_loaded:.2f}s", flush=True)

Importing UMAP...
Loading years, layer 19 for UMAP...
Loaded in 0.01s; fitting 3D UMAP...
UMAP fit finished in 3.86s


In [ ]:
# The PCA plot supplies matching label colors and prompt hover text.
for result in (umap_result, results[UMAP_LAYER]):
    if result["activation_dir"] != str(ACTIVATION_DIR) or result["subset"] != SUBSET or not np.array_equal(
        result["source_row_ids"], selected["source_row_id"].to_numpy()
    ):
        raise ValueError("The selection changed: rerun its PCA, PCA plot, and UMAP cells")

embedding = umap_result["scores"]
umap_fig = go.Figure(go.Scatter3d(
    x=embedding[:, 0], y=embedding[:, 1], z=embedding[:, 2],
    mode="markers", marker=marker, text=hover_text,
    hovertemplate="%{text}<br>UMAP 1=%{x:.3f}<br>UMAP 2=%{y:.3f}<br>UMAP 3=%{z:.3f}<extra></extra>",
))
umap_fig.update_layout(
    title=f"{umap_result['subset']} · last token · layer {umap_result['layer']:02d} · 3D UMAP",
    height=720, margin={"l": 0, "r": 0, "b": 0, "t": 60},
    scene={"xaxis_title": "UMAP 1", "yaxis_title": "UMAP 2",
           "zaxis_title": "UMAP 3", "aspectmode": "data"},
)
umap_fig.show()